<a href="https://colab.research.google.com/github/faheemjabbar39-hub/llm-knowledge-integration/blob/main/notebooks/run_baseline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline run (Base strategy)
**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order (click ▶ or press Shift+Enter).

In [ ]:
# Check that a GPU is attached. You should see 'Tesla T4'.
!nvidia-smi

## 1. Settings
Use `mistral7b` while your Llama access is pending. Change to `llama8b` once Meta approves.

In [ ]:
MODEL = "mistral7b"   # options: mistral7b, llama8b  (llama8b baseline already done)

## 2. Get the code from your private GitHub repo
Paste a GitHub token when asked (github.com/settings/tokens → Generate new token (classic) → tick **repo** → Generate). It is hidden as you type and not saved in the notebook.

In [ ]:
import getpass, os
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
gh_token = getpass.getpass("GitHub token: ")
if not os.path.exists("/content/project"):
    !git clone -q https://{gh_token}@{REPO} /content/project
    !git -C /content/project remote set-url origin https://{REPO}   # don't keep the token on disk
else:
    !git -C /content/project pull -q https://{gh_token}@{REPO} main
del gh_token
%cd /content/project
!git log --oneline -1

## 3. Install packages (~3 minutes)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pytest -q

## 4. Log in to Hugging Face
Paste your Hugging Face **Read** token (starts with `hf_`).

In [ ]:
from huggingface_hub import login
login()

## 5. Download the benchmarks and split them into train / test

In [ ]:
!python scripts/01_prepare_benchmarks.py
!wc -l data/processed/*.jsonl

## 6. Smoke test: 20 questions per dataset (~5 minutes incl. model download)

In [ ]:
!python scripts/05_run_eval.py --model $MODEL --strategy base --limit 20
!python scripts/06_analyze.py

## 7. Full baseline (~1 hour on a T4)
If Colab disconnects, run cells 2-4 again and then this cell: it continues where it stopped.

In [ ]:
!python scripts/05_run_eval.py --model $MODEL --strategy base

## 8. Results

In [ ]:
# If this is a new Colab session, first restore earlier results from Drive:
# from google.colab import drive; drive.mount('/content/drive'); !cp -n /content/drive/MyDrive/kis_results/raw/* results/raw/
!python scripts/06_analyze.py --gpu-usd-per-hour 0.35
!python scripts/08_plot_results.py
import pandas as pd
pd.read_csv('results/summary.csv')

## Charts
One panel per model, one colour per strategy. They grow automatically as you run more models and strategies.

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

## 9. Save results to Google Drive (Colab deletes files when the session ends)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kis_results
!cp -r results/* /content/drive/MyDrive/kis_results/
!ls /content/drive/MyDrive/kis_results/raw

## 10. Upload results to GitHub
Paste your GitHub token again. This saves the results CSVs, summary and charts to the repo with a commit message.

In [ ]:
import getpass, datetime
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
tok = getpass.getpass("GitHub token: ")
!git config user.name "Faheem Jabbar"
!git config user.email "faheemjabbar39@gmail.com"
!git add results/
msg = f"Results: {MODEL} ({datetime.date.today()})"
!git commit -q -m "{msg}" || echo "nothing new to commit"
!git pull -q --rebase -X theirs https://{tok}@{REPO} main
!git push -q https://{tok}@{REPO} HEAD:main && echo "Uploaded to GitHub"
del tok